conda install -c conda-forge nb_conda_kernels

(Markdown): 整体环境准备


# 多实例学习 (MIL) 疾病预测流水线

本 Notebook 负责串联整个端到端的训练流程：
1. **特征提取**：使用预训练大模型（如 DINOv2）将原始患者图像提取为一维特征向量，并保存在 Excel 中。
2. **特征处理**：将 Excel 表格按患者 ID 拆分为独立的 PyTorch 张量 (`.pt`) 文件，加速后续读取。
3. **模型训练**：使用多实例学习 (MIL) 的注意力机制和分类头，基于离线特征进行快速训练和 5 折交叉验证。


安装依赖
# 确保安装了所有必须的第三方库
!pip install pandas openpyxl scikit-learn libauc iterative-stratification timm

第一步：离线特征提取
## 第一步：从原始图像中提取特征 (Offline Feature Extraction)
调用 `extract_instance_features.py`，遍历 `data_root` 下的所有患者图片。
* **--encoder**: 使用的网络（如 `vit_base_patch14_dinov2` 或 `efficientnet_b0`）
* **--freeze_encoder 1**: 冻结编码器权重，纯作特征提取器
* **--out_xlsx**: 提取的特征将全部保存到此 Excel 文件中

执行特征提取

%run extract_instance_features.py \
  --data_root ../newData \
  --labels_csv ./labels03.xlsx \
  --label_cols 代谢慢病 \
  --max_images -1 \
  --encoder vit_base_patch14_dinov2 \
  --pretrained 1 \
  --weights_path ../weight/dinov2_vitb14_pretrain.pth \
  --freeze_encoder 1 \
  --out_xlsx ./outputs/encoder_instance_features.xlsx

读取文件的前5行

In [1]:
import pandas as pd
#读取特征提取文件的前10行
df = pd.read_excel("./outputs/encoder_instance_features.xlsx",nrows=10)
print(df)

   patient_id            image_path    feat_0    feat_1    feat_2    feat_3  \
0           1  ../newData/1/001.jpg  2.713709  2.997745  1.139498 -2.603614   
1           1  ../newData/1/002.jpg  3.880903  3.372237  0.804817 -2.222047   
2           1  ../newData/1/003.jpg  1.845182  3.969146  2.279546 -1.452059   
3           1  ../newData/1/004.jpg  1.514021  3.676345  2.079935 -1.083477   
4           1  ../newData/1/005.jpg  2.606607  3.035503  2.478067 -2.373037   
5           1  ../newData/1/006.jpg  2.739537  3.742360  1.649610 -1.254980   
6           1  ../newData/1/007.jpg  2.223447  3.150883  3.110158 -2.212480   
7           1  ../newData/1/008.jpg  3.159253  3.073846  0.708921 -1.192649   
8           1  ../newData/1/010.jpg  2.089792  1.725832  2.073613 -1.543648   
9           1  ../newData/1/011.jpg  3.092132  2.828193  1.644440 -2.423040   

     feat_4    feat_5    feat_6    feat_7  ...  feat_758  feat_759  feat_760  \
0  3.082359  0.725815  0.292928 -0.130321  ... -2.

In [4]:
import pandas as pd
df = pd.read_excel("./outputs/encoder_instance_features.xlsx")
print(df.shape)

(41980, 770)


第二步：特征格式转换
## 第二步：将 Excel 特征转换为 `.pt` 格式
为了在训练时能够快速加载并进行 Batch 拼接，我们将刚才生成的巨大 Excel 表格，按 `patient_id` 拆分成一个个独立的 `.pt` 文件，统一存放在 `encoder_features/` 目录下。

In [5]:
import pandas as pd
import torch
import os

print("⏳ 正在读取庞大的特征 Excel 文件，请稍候...")
# 读取刚才生成的 excel 文件
df_features = pd.read_excel("./outputs/encoder_instance_features.xlsx")

# 筛选出所有以 'feat_' 开头的特征列
feature_cols = [c for c in df_features.columns if c.startswith("feat_")]

# 创建存放 pt 文件的文件夹
save_dir = "encoder_features"
os.makedirs(save_dir, exist_ok=True)

print(f"🔄 开始将特征按患者拆分并保存到 {save_dir}/ 目录...")

# 按患者 ID 进行分组
for pid, group in df_features.groupby("patient_id"):
    # 提取特征矩阵，形状为 [N, D] (N为图像数，D为特征维度，如768)
    feats = group[feature_cols].values
    feats_tensor = torch.tensor(feats, dtype=torch.float32)
    
    # 保存为单独的 pt 文件
    torch.save(
        {"patient_id": pid, "feats": feats_tensor},
        f"{save_dir}/{pid}.pt"
    )

print("✅ 特征转换彻底完成！")




⏳ 正在读取庞大的特征 Excel 文件，请稍候...
🔄 开始将特征按患者拆分并保存到 encoder_features/ 目录...
✅ 特征转换彻底完成！


第三步：启动 MIL 模型训练
## 第三步：训练注意力层与分类器 (MIL Training)
读取上一步生成的 `.pt` 文件，基于旧版代码设定的超参数进行训练。
* **--max_feats 30**: 强制将每个患者的特征对齐到 30 张图（超出的截断，不足的补零/随机采样），从而支持 `batch_size=8`。
* **--use_combined_loss**: 开启 BCE + AUCMLoss 混合损失函数。

执行训练
# 核心超参数（lr, epochs, batch_size 等）与旧代码严格对齐

In [1]:
%run train_log.py \
  --feat_dir ./encoder_features \
  --labels_csv ./labels03.xlsx \
  --label_cols 代谢慢病 \
  --epochs 100 \
  --batch_size 8 \
  --lr 5e-4 \
  --weight_decay 1e-4 \
  --max_feats 30 \
  --folds 5 \
  --num_workers 0 \
  --instance_strategy random \
  --architecture attention \
  --use_combined_loss \
  --auc_weight 0.5 \
  --seed 42

===== TRAINING START =====
feat_dir: ./encoder_features
labels_csv: ./labels03.xlsx
label_cols: ['代谢慢病']
epochs: 100
batch_size: 8
lr: 0.0005
weight_decay: 0.0001
max_feats: 30
folds: 5
num_workers: 0
instance_strategy: random
architecture: attention
use_combined_loss: True
auc_weight: 0.5
seed: 42

Single-label task: StratifiedKFold
Using device: cuda

===== Fold 1/5 =====
Epoch 001: loss=0.4793  AUROC=0.5248  AUPRC=0.4841  Sens@95Spec=0.0976  Brier=0.2440  ✓ NEW BEST
Epoch 002: loss=0.4537  AUROC=0.5385  AUPRC=0.4736  Sens@95Spec=0.0732  Brier=0.2476  ✓ NEW BEST
Epoch 003: loss=0.4596  AUROC=0.5407  AUPRC=0.4950  Sens@95Spec=0.1220  Brier=0.2458  ✓ NEW BEST
Epoch 004: loss=0.4488  AUROC=0.5603  AUPRC=0.4884  Sens@95Spec=0.0976  Brier=0.2478  ✓ NEW BEST
Epoch 005: loss=0.4525  AUROC=0.5627  AUPRC=0.4844  Sens@95Spec=0.0732  Brier=0.2441  ✓ NEW BEST
Epoch 006: loss=0.4364  AUROC=0.5677  AUPRC=0.5125  Sens@95Spec=0.1220  Brier=0.2515  ✓ NEW BEST
Epoch 007: loss=0.4369  AUROC=0.5420  AUP

In [7]:
pip install libauc


  Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 33.4 MB/s  0:00:02m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 38.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 31.9 MB/s  0:00:00
Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl (15 kB)
Using cached aiosignal-1.4.0-py3-none-any.whl (7.5 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16/16 [libauc]15/16 [libauc]eometric]
Note: you may need to restart the kernel to use updated packages.
